# Best-Methods Figure

This notebook generates figures 5 and 19 for the Parameter Setting paper. Figure
5 is the "best-methods figure", showing the methods which gave the best energies
on a specific set of target instances. Figure 19 contains a CDF of the
"advantage" provided by Linear Ramp* over Interp.*, which are the methods which
are "best" most often.


In [2]:
import warnings
from collections import defaultdict
from itertools import product
from pathlib import Path
from typing import Any, Literal, cast

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
from tqdm.auto import tqdm

import qaoa_parameter_setting.utils as utils
import qaoa_parameter_setting.utils.database as database
from qaoa_parameter_setting.utils.types import (
    Depth,
    EvaluationType,
    GraphKey,
    MethodConfigJSON,
)

mpl.use("pdf")

c:\Users\joely\anaconda3\envs\qaoa_pipeline\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Styling for the paper

In [3]:
# Configure matplotlib to use LaTeX for text rendering and set font sizes
plt.rcParams["text.usetex"] = True
plt.rcParams["legend.fontsize"] = "small"
plt.rcParams["legend.columnspacing"] = 1
plt.rcParams["axes.titlesize"] = "small"
plt.rcParams["axes.labelsize"] = "small"
plt.rcParams["figure.labelsize"] = "small"
plt.rcParams["xtick.labelsize"] = "small"
plt.rcParams["ytick.labelsize"] = "small"

# We use Paul Tol's bright colour scheme
# Define color mapping for each method variant (with optimization markers * and †)
method_to_colour = {
    "Fixed Angles*": "#4477AA",
    "Fixed Angles": "#4477AA",
    "Fixed Angles†": "#4477AA",
    "Fourier*": "#EE6677",
    "Interp.*": "#228833",
    "Linear Ramp*": "#CCBB44",
    "Linear Ramp": "#CCBB44",
    "Linear Ramp†": "#CCBB44",
    "Recursive TS*": "#66CCEE",
    "TQA*": "#AA3377",
    "TQA": "#AA3377",
    "TQA†": "#AA3377",
    "Parameter Transfer*": "#BBBBBB",
}
# Define marker shapes for different evaluation methods
evaluation_markers = {
    "SV": "o",
    "MPS (Quimb)": "P",
    "MPS (Aer)": "^",
    "PP": "s",
}


def style_scatter(method: str, evaluation: str) -> dict:
    """Return a style dictionary suitable for :fun:`matplotlib.pyplot.scatter`.

    Args:
        method: The method label, with ``"*"`` or ``"†"`` if the method is
            optimised or unoptimised.
        evaluation: The energy evaluation method label, e.g., ``"SV"``, ``"MPS
            (Aer)"``, ``"MPS (Quimb)"``, or ``"PP"``.

    Returns:
        A dictionary of kwargs appropriate for matplotlib's scatter function.
    """
    # Configure marker styling based on optimization status:
    # - "*" (optimized angles): black edge, colored fill
    # - "†" (unoptimized): colored edge, white fill
    # - neither: no edge, colored fill
    return {
        "edgecolor": "k"
        if "*" in method
        else (method_to_colour[method] if "†" in method else "None"),
        "marker": evaluation_markers[evaluation],
        "color": method_to_colour[method] if "†" not in method else "None",
        "facecolor": "w" if "†" in method else method_to_colour[method],
    }


def style_plot(method: str, evaluation: str) -> dict:
    """Return a style dictionary suitable for :fun:`matplotlib.pyplot.plot`.

    Args:
        method: The method label, with ``"*"`` or ``"†"`` if the method is
            optimised or unoptimised.
        evaluation: The energy evaluation method label, e.g., ``"SV"``, ``"MPS
            (Aer)"``, ``"MPS (Quimb)"``, or ``"PP"``.

    Returns:
        A dictionary of kwargs appropriate for matplotlib's plot function.
    """
    # Start with scatter styling
    kwargs = style_scatter(method=method, evaluation=evaluation)
    # Rename entries from scatter() kwargs to plot() kwargs
    kwargs["markerfacecolor"] = kwargs.pop("facecolor")
    kwargs["markeredgecolor"] = kwargs.pop("edgecolor")
    # Set line color to method color
    kwargs["color"] = method_to_colour[method]
    return kwargs


def figsize(
    aspect_ratio: float = 1.0, two_columns: bool = False
) -> tuple[float, float]:
    # Calculate figure dimensions based on column layout
    if two_columns:
        width = 7.0
    else:
        width = 3.4
    height = width / aspect_ratio
    return (width, height)


## Target Instances

In [4]:
# Here we hard-code which instances we want, so we are consistent between all
# bars and axes in the figure.

# idx chosen to reduce the number of new experiments to run, based on already
# existing runs.
idx = "001"
SMALL_INSTANCES = {
    # 10x 20-node 20-percent Erdös-Rényi graphs
    "000_20nodes_erdosrenyi20percent.json",
    "001_20nodes_erdosrenyi20percent.json",
    "002_20nodes_erdosrenyi20percent.json",
    "003_20nodes_erdosrenyi20percent.json",
    "004_20nodes_erdosrenyi20percent.json",
    "005_20nodes_erdosrenyi20percent.json",
    "006_20nodes_erdosrenyi20percent.json",
    "007_20nodes_erdosrenyi20percent.json",
    "008_20nodes_erdosrenyi20percent.json",
    "009_20nodes_erdosrenyi20percent.json",
    # 10x 1-by-2 21-node Heavy-Hex graphs
    "000_1_2_heavyhex_21nodes_weighted.json",
    "001_1_2_heavyhex_21nodes_weighted.json",
    "002_1_2_heavyhex_21nodes_weighted.json",
    "003_1_2_heavyhex_21nodes_weighted.json",
    "004_1_2_heavyhex_21nodes_weighted.json",
    "005_1_2_heavyhex_21nodes_weighted.json",
    "006_1_2_heavyhex_21nodes_weighted.json",
    "007_1_2_heavyhex_21nodes_weighted.json",
    "008_1_2_heavyhex_21nodes_weighted.json",
    "009_1_2_heavyhex_21nodes_weighted.json",
    # 1x 20-node line-to-full graph per 1 to 10 swap layers, for total of 10x graphs
    f"{idx}_20nodes_1swap_layers.json",
    f"{idx}_20nodes_2swap_layers.json",
    f"{idx}_20nodes_3swap_layers.json",
    f"{idx}_20nodes_4swap_layers.json",
    f"{idx}_20nodes_5swap_layers.json",
    f"{idx}_20nodes_6swap_layers.json",
    f"{idx}_20nodes_7swap_layers.json",
    f"{idx}_20nodes_8swap_layers.json",
    f"{idx}_20nodes_9swap_layers.json",
    f"{idx}_20nodes_10swap_layers.json",
    # 1x 20-node random-regular graph per degree d=3,4,5,6,7,8,9 for a total of 7x graphs
    f"{idx}_20nodes_random3regular.json",
    f"{idx}_20nodes_random4regular.json",
    f"{idx}_20nodes_random5regular.json",
    f"{idx}_20nodes_random6regular.json",
    f"{idx}_20nodes_random7regular.json",
    f"{idx}_20nodes_random8regular.json",
    f"{idx}_20nodes_random9regular.json",
}
LARGE_INSTANCES = {
    # 10x 50-node 20-percent Erdös-Rényi graphs
    "000_50nodes_erdosrenyi20percent.json",
    "001_50nodes_erdosrenyi20percent.json",
    "002_50nodes_erdosrenyi20percent.json",
    "003_50nodes_erdosrenyi20percent.json",
    "004_50nodes_erdosrenyi20percent.json",
    "005_50nodes_erdosrenyi20percent.json",
    "006_50nodes_erdosrenyi20percent.json",
    "007_50nodes_erdosrenyi20percent.json",
    "008_50nodes_erdosrenyi20percent.json",
    "009_50nodes_erdosrenyi20percent.json",
    # 10x 144-node Heavy-Hex graphs
    "000_7_3_heavyhex_144nodes_weighted.json",
    "001_7_3_heavyhex_144nodes_weighted.json",
    "002_7_3_heavyhex_144nodes_weighted.json",
    "003_7_3_heavyhex_144nodes_weighted.json",
    "004_7_3_heavyhex_144nodes_weighted.json",
    "005_7_3_heavyhex_144nodes_weighted.json",
    "006_7_3_heavyhex_144nodes_weighted.json",
    "007_7_3_heavyhex_144nodes_weighted.json",
    "008_7_3_heavyhex_144nodes_weighted.json",
    "009_7_3_heavyhex_144nodes_weighted.json",
    # 1x 100-node line-to-full graph per 1 to 10 swap layers, for total of 10x graphs
    f"{idx}_100nodes_1swap_layers.json",
    f"{idx}_100nodes_2swap_layers.json",
    f"{idx}_100nodes_3swap_layers.json",
    f"{idx}_100nodes_4swap_layers.json",
    f"{idx}_100nodes_5swap_layers.json",
    f"{idx}_100nodes_6swap_layers.json",
    f"{idx}_100nodes_7swap_layers.json",
    f"{idx}_100nodes_8swap_layers.json",
    f"{idx}_100nodes_9swap_layers.json",
    f"{idx}_100nodes_10swap_layers.json",
    # 1x 100-node random-regular graph per degree d=3,4,5,6,7,8,9, for a total of 7x graphs
    f"{idx}_100nodes_random3regular.json",
    f"{idx}_100nodes_random4regular.json",
    f"{idx}_100nodes_random5regular.json",
    f"{idx}_100nodes_random6regular.json",
    f"{idx}_100nodes_random7regular.json",
    f"{idx}_100nodes_random8regular.json",
    f"{idx}_100nodes_random9regular.json",
}

chosen_instances: dict[EvaluationType, set[GraphKey] | dict[bool, set[str]]] = {
    "SV": SMALL_INSTANCES,
    "PP": LARGE_INSTANCES,
    "MPS": {False: LARGE_INSTANCES, True: LARGE_INSTANCES},
}
ALL_CHOSEN_INSTANCES = SMALL_INSTANCES.union(LARGE_INSTANCES)

In [5]:
def ignore_these_files(filename: str, results: dict[str, Any]) -> bool:
    """Ignore Parameter Transfer and unoptimised Linear Ramp results for this figure."""

    # If we have Parameter Transfer or Fourier refined, ignore.
    if "PT_" in filename or "refined" in filename:
        return True

    # NOTE: The following filtering is made on "old" filenames.
    # If we have Linear Ramp without angle optimisation, ignore. This resolves
    # some bond-dimension inconsistencies between _opt and _angle_opt LR
    # results. These are instead extracted from _angle_opt data.
    if (
        "_LR_" in filename
        and "_opt" in filename.lower()
        and not ("angle_opt" in filename.lower() or "angleopt" in filename.lower())
    ):
        # We have LR without angle optimisation
        if "angle" in filename:
            print(f"WARNING: Ignoring {filename!r}")
        return True
    return False


db_filename: str | None = None
if db_filename is not None:
    # If we're loading from a saved file.
    db = database.ResultsDatabase(db_filename)
else:
    db = db = database.ResultsDatabase()
    # Add date
    for _folder in tqdm(
        [
            "../data/training/random_regular",
            "../data/training/heavy_hex",
            "../data/training/line_to_full",
            "../data/training/erdos_renyi",
        ]
    ):
        db.add_data(_folder, ignore_file_function=ignore_these_files)
db = db.filter_by(instance_filter=chosen_instances)

100%|██████████| 4/4 [21:09<00:00, 317.34s/it]


In [6]:
# These are all methods present in the data.
db.print_methods_by_evaluation()

      MPS (Aer)        |     MPS (Quimb)     |         PP         |         SV        
--------------------------------------------------------------------------------------
FA_MPSAer_no_opt.json  | FA_MPS_no_opt.json  | FA_PP_no_opt.json  | FA_SV_no_opt.json 
FA_MPSAer_opt.json     | FA_MPS_opt.json     | FA_PP_opt.json     | FA_SV_opt.json    
F_MPSAer_opt.json      | F_MPS_opt.json      | F_PP_opt.json      | F_SV_opt.json     
I_MPSAer_opt.json      | I_MPS_opt.json      | I_PP_opt.json      | I_SV_opt.json     
LR_MPSAer.json         | LR_MPS.json         | LR_PP.json         | LR_SV.json        
LR_MPSAer_no_opt.json  | LR_MPS_no_opt.json  | LR_PP_no_opt.json  | LR_SV_no_opt.json 
LR_MPSAer_opt.json     | LR_MPS_opt.json     | LR_PP_opt.json     | LR_SV_opt.json    
RTS_MPSAer_opt.json    | RTS_MPS_opt.json    | RTS_PP_opt.json    | RTS_SV_opt.json   
TQA_MPSAer.json        | TQA_MPS.json        | TQA_PP.json        | TQA_SV.json       
TQA_MPSAer_no_opt.json | TQA_MPS_no_opt.jso

## Best-Methods Figure


### Get Best-Methods Dataframe


In [7]:
# Get best methods per instance, i.e., instance-depth pairs. As we already
# filtered by the instances, this will ensure we only have the best-method per
# instance in `chosen_instances`.

# We ignore the warnings about missing min-max cut data. We didn't add those
# files to the database as the best-methods figure doesn't use approximation
# ratios AND we can determine the best methods using the energies.
warnings.filterwarnings("ignore", "Missing min-max cut data for instance")

# Create the best-methods dataframe.
df_best = db.only_best_parameters("instance").to_dataframe()
df_best

,instance,num_nodes,graph_type,trainer_config,method,depth,energy,trainer,evaluation,evaluation_label,...,metadata,result_index,run_datetime,result_key_index,approximation_ratio,mps_bond_dimension,mps_threshold,pp_max_weight,pp_min_abs_coeff,fa_degree
0,001_20nodes_random3regular.json,20,random_regular,FA_SV_opt.json,FA_opt.json,10,10.380828,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 13, 'evaluator':...",0,2025-09-01 15:07:45,1.0,NaN,NaN,NaN,NaN,NaN,None
1,001_20nodes_random3regular.json,20,random_regular,FA_SV_opt.json,FA_opt.json,1,5.156636,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 13, 'evaluator':...",0,2025-09-13 17:13:56,1.0,NaN,NaN,NaN,NaN,NaN,None
2,001_20nodes_random3regular.json,20,random_regular,FA_SV_opt.json,FA_opt.json,3,8.330960,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 13, 'evaluator':...",0,2025-09-13 20:54:55,1.0,NaN,NaN,NaN,NaN,NaN,None
3,001_20nodes_random3regular.json,20,random_regular,FA_SV_opt.json,FA_opt.json,4,9.059836,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 13, 'evaluator':...",0,2025-09-13 21:00:11,1.0,NaN,NaN,NaN,NaN,NaN,None
4,001_20nodes_random3regular.json,20,random_regular,FA_SV_opt.json,FA_opt.json,5,9.528705,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 13, 'evaluator':...",0,2025-09-13 21:04:04,1.0,NaN,NaN,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1475,008_20nodes_erdosrenyi20percent.json,20,erdos_renyi,LR_SV_opt.json,LR_opt.json,7,11.660903,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 33, 'evaluator':...",0,2026-04-28 10:43:35,1.0,NaN,NaN,NaN,NaN,NaN,None
1476,008_20nodes_erdosrenyi20percent.json,20,erdos_renyi,LR_SV_opt.json,LR_opt.json,8,11.838120,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 33, 'evaluator':...",0,2026-04-28 11:04:57,1.0,NaN,NaN,NaN,NaN,NaN,None
1477,008_20nodes_erdosrenyi20percent.json,20,erdos_renyi,LR_SV_opt.json,LR_opt.json,10,12.257593,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 33, 'evaluator':...",0,2026-04-28 11:57:49,1.0,NaN,NaN,NaN,NaN,NaN,None
1478,008_20nodes_erdosrenyi20percent.json,20,erdos_renyi,FA_SV_opt.json,FA_opt.json,4,10.342585,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 33, 'evaluator':...",0,2026-04-02 16:33:30,1.0,NaN,NaN,NaN,NaN,NaN,None


### Plot Best-Methods Figure


In [8]:
fig, axes = plt.subplots(
    2,
    2,
    figsize=figsize(1.0),
    sharey=True,
    sharex=True,
    dpi=300,
    gridspec_kw=dict(hspace=0.25, wspace=0.1),
)
handles, labels = None, None

# We only label methods which are best at least once.
training_methods = list(str(x) for x in sorted(df_best["method_label"].unique()) if "refined" not in x)
custom_palette = [method_to_colour[method] for method in training_methods]

for ax, (evaluation, with_aer) in zip(
    axes.flatten(), [("SV", False), ("PP", False), ("MPS", False), ("MPS", True)]
):
    # Set titles, x-ticks, and x limits.
    ax.set_title(
        f"{evaluation} {('(Aer)' if with_aer else '(Quimb)') if evaluation == 'MPS' else ''}"
    )
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1, offset=1))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))
    ax.set_xlim(1.5, 10.5)

    # Get the data for this evaluation method.
    # We use cast() so we don't get type-hint errors.
    sub_data = cast(
        pd.DataFrame,
        df_best[
            (df_best["evaluation"] == evaluation)
            & (df_best["with_aer"] == with_aer)
            & (df_best["depth"] > 1)
        ],
    )

    # Print error message if there are no results.
    if len(sub_data) == 0:
        print(f"No data for evaluation={evaluation} and with_aer={with_aer}")
        continue

    # Plot using Seaborn as matplotlib doesn't seem to support stacked bars.
    _ = sns.histplot(
        data=sub_data,
        x="depth",
        hue="method_label",
        multiple="fill",
        stat="proportion",
        ax=ax,
        # Use bins that are offset, so we catch the correct depths.
        bins=np.arange(1.5, 11, 1),
        hue_order=training_methods,
        legend=False,
        palette=custom_palette,
    )

    # Remove xlabel and ylabel as we use supxlabel and supylabel. We do this
    # after sns.histplot as Seaborn also sets the axis labels.
    ax.set_xlabel("")
    ax.set_ylabel("")

# Set figure labels.
fig.supylabel("Proportion", x=0)
fig.supxlabel("Depth $p$", y=0)

# Create legend.
legend_entries = [
    (
        Patch(facecolor=hue, edgecolor="k"),
        utils.labels.format_method_label_to(label, "latex"),
    )
    for hue, label in zip(
        sns.color_palette(custom_palette)
        if isinstance(custom_palette, str)
        else custom_palette,
        training_methods,
    )
]
fig.legend(
    *zip(
        *np.fromiter(legend_entries, dtype=object)
        .reshape((2, -1), order="C")
        .reshape((-1,), order="F")
        .tolist()
    ),
    loc="upper center",
    bbox_to_anchor=(0.5, 0, 0, 0),
    ncol=3,
    frameon=False,
)
# plt.tight_layout()
plt.savefig("figures/figure5.pdf", bbox_inches="tight", dpi=300)

## Get Missing Configurations


Compute the missing configurations as the difference between the target
configurations and the actual configurations, ignoring "failed" runs as loaded
from `failed_runs_best_methods.json`. The JSON file contains Fixed Angles
configs where the instance has no entry in the fixed-angles database.

In [9]:
# Failed runs are those which did not return a valid results JSON file. We mark
# why in in `failed_runs.json`
failed_runs_filename = Path("figures/extra_info/figure5_failed_runs.json")
failed_runs: dict[GraphKey, dict[MethodConfigJSON, dict[Depth, str]]]
if failed_runs_filename.exists():
    failed_runs = db.load_failed_configs_from_json(
        failed_runs_filename,
        methods_is_old=True,
    )
else:
    failed_runs = defaultdict(lambda: defaultdict(dict))


In [10]:
from qaoa_parameter_setting.utils.labels import sanitize_trainer_config


def sanitize_trainer_config_list(
    methods: list[str],
    is_old: bool = True,
) -> list[MethodConfigJSON]:
    return [sanitize_trainer_config(_method, is_old=is_old) for _method in methods]


# These are the methods we would like to include in our figure. Some are
# 'virtual' _no_opt methods as no method JSON file exists, but we extract the
# data from an intermediate run of an _opt-method run.
target_methods: dict[
    EvaluationType, list[MethodConfigJSON] | dict[bool, list[MethodConfigJSON]]
] = {
    "MPS": {
        True: sanitize_trainer_config_list(
            [
                "FA_MPSAer_no_opt.json",
                "FA_MPSAer_opt.json",
                "F_MPSAer.json",
                "I_MPSAer.json",
                "LR_MPSAer_opt.json",
                "LR_MPSAer_angle_opt.json",
                "TQA_MPSAer_no_opt.json",
                "TQA_MPSAer_opt.json",
            ],
        ),
        False: sanitize_trainer_config_list(
            [
                "FA_MPS_no_opt.json",
                "FA_MPS_opt.json",
                "F_MPS.json",
                "I_MPS.json",
                "LR_MPS_opt.json",
                "LR_MPS_angle_opt.json",
                "RTS_MPS.json",
                "TQA_MPS_no_opt.json",
                "TQA_MPS_opt.json",
            ],
        ),
    },
    "PP": sanitize_trainer_config_list(
        [
            "FA_PP_no_opt.json",
            "FA_PP_opt.json",
            "F_PP.json",
            "I_PP.json",
            "LR_PP_angle_opt.json",
            "LR_PP_opt.json",
            # We don't want Parameter Transfer at all.
            # "PT_PP_AAAM.json",
            "RTS_PP.json",
            "TQA_PP_no_opt.json",
            "TQA_PP_opt.json",
        ],
    ),
    "SV": sanitize_trainer_config_list(
        [
            "FA_SV_no_opt.json",
            "FA_SV_opt.json",
            "F_SV.json",
            "I_SV.json",
            "LR_SV_angle_opt.json",
            "TQA_SV_no_opt.json",
            "TQA_SV_opt.json",
            "TS_SV.json",
        ],
    ),
}

all_methods: list[MethodConfigJSON] = list(
    set(
        method
        for methods in target_methods.values()
        for method in (
            methods
            if isinstance(methods, list)
            else [m for sublist in methods.values() for m in sublist]
        )
    )
)

In [11]:
missing: dict[str, dict[bool, dict[str, list[int]]]] = {}
failed_records = []
for evaluation, with_aer in [
    ("SV", False),
    ("PP", False),
    ("MPS", False),
    ("MPS", True),
]:
    for __inst, __inst_data in failed_runs.items():
        if evaluation == "MPS" and __inst not in chosen_instances[evaluation][with_aer]:
            continue
        elif evaluation != "MPS" and __inst not in chosen_instances[evaluation]:
            continue
        for __method, __method_data in __inst_data.items():
            if (
                utils.labels.trainer_config_to_evaluation(__method) != evaluation
                or utils.labels.method_uses_aer(__method) != with_aer
            ):
                continue
            if utils.labels.trainer_config_to_method_label(__method) != "Fixed Angles*":
                continue
            if evaluation not in missing:
                missing[evaluation] = {}
            if with_aer not in missing[evaluation]:
                missing[evaluation][with_aer] = {}
            if __inst not in missing[evaluation][with_aer]:
                missing[evaluation][with_aer][__inst] = []
            missing[evaluation][with_aer][__inst].extend(list(__method_data.keys()))
            for _depth in __method_data.keys():
                failed_records.append(
                    {
                        "evaluation": evaluation,
                        "with_aer": with_aer,
                        "instance": __inst,
                        "method": __method,
                        "depth": _depth,
                    }
                )

In [12]:
failed_df = pd.DataFrame.from_records(failed_records)
failed_df = failed_df[failed_df["depth"] > 1]
(
    failed_df.pivot_table(
        values="instance",
        index=["evaluation", "with_aer"],
        # columns="depth",
        aggfunc="count",
    )
)

instance
evaluation with_aer          
MPS        False          265
           True           265
PP         False          265
SV         False          237

In [13]:
# Get missing configs. We make the database's life easier by only storing the
# best-parameters per config, i.e., the best energy per instance-method-depth
# tuples.
missing_configs: dict[
    EvaluationType | tuple[Literal["MPS"], bool],
    dict[MethodConfigJSON, dict[Depth, set[GraphKey]]],
] = db.only_best_parameters("config").get_missing_configs(
    target_methods=target_methods,
    target_instances=chosen_instances,
    target_depths=range(2, 11),
    failed_configs=failed_runs,
    with_derived_configs=False,
)

if missing_configs:
    # Convert the missing configs into a dataframe.
    db_missing_configs_data = []
    for eval_key, method_dict in missing_configs.items():
        if isinstance(eval_key, tuple):
            evaluation, with_aer = eval_key
        else:
            evaluation = eval_key
            with_aer = None

        for method_config, depth_dict in method_dict.items():
            for depth, instance_set in depth_dict.items():
                for instance_name in instance_set:
                    db_missing_configs_data.append(
                        {
                            # Helper fields
                            "evaluation": evaluation,
                            "with_aer": with_aer,
                            "graph_type": utils.instance.graph_type(instance_name),
                            "evaluation_label": utils.labels.trainer_config_to_evaluation_label(
                                method_config
                            ),
                            "method_label": utils.labels.trainer_config_to_method_label(
                                method_config
                            ),
                            # Config
                            "method": method_config,
                            "depth": depth,
                            "instance": instance_name,
                        }
                    )

    df_missing_configs = pd.DataFrame(db_missing_configs_data)
    if not df_missing_configs.empty:
        df_missing_configs = df_missing_configs.set_index(
            ["evaluation", "with_aer", "method", "depth"]
        )
    df_missing_configs = df_missing_configs.reset_index().sort_values(
        ["evaluation", "with_aer", "method", "depth", "instance"]
    )
    # Save to a csv for easier referencing
    df_missing_configs.to_csv("figures/extra_info/figure5_missing_configs.csv")
    display(df_missing_configs)
else:
    # Write an empty CSV file
    with open("figures/extra_info/figure5_missing_configs.csv", "w") as f:
        f.write(
            ",evaluation,with_aer,method,depth,graph_type,evaluation_label,method_label,instance\n"
        )
    print("No missing configs.")

No missing configs.


### Missing Configurations Table


Create a table of the missing configurations.

In [14]:
# Create multi-index of target evaluation and method labels.
row_index = pd.MultiIndex.from_tuples(
    [
        (
            evaluation_label,
            method_label,
        )
        for evaluation_label, method_label in product(
            ["MPS (Aer)", "MPS (Quimb)", "PP", "SV"],
            sorted(
                set(
                    [
                        utils.labels.trainer_config_to_method_label(m)
                        for m in all_methods
                    ]
                )
            ),
        )
    ],
)

In [15]:
if missing_configs:
    pivot_missing = df_missing_configs.pivot_table(
        values="instance",
        index=["evaluation_label", "method_label"],
        columns=["graph_type", "depth"],
        aggfunc="count",
    )
    pivot_missing = pivot_missing.reindex(index=row_index)
    display("Missing Runs for Best-Methods Figure")
    display(
        pivot_missing.style.format(precision=0, na_rep=" ")
        .background_gradient("RdYlGn_r", axis=None)  # pyright: ignore[reportAttributeAccessIssue]
        .highlight_null("transparent")
    )
else:
    print("No missing configs.")

No missing configs.


# Distribution of Advantage for Linear Ramp* vs Interp.*


We want to compare the performance of Linear Ramp* and Interp.* on instances where the best method was one of them. For this we do the following:
1. Identify all instances of interest from the best-methods figure. We constrain this to depths $P=2,3,\ldots,10$ as this is the domain of the best-methods figure.
2. We build a dataframe of the best Linear Ramp* and Interp.* runs for those instances.
3. The relative advantage is computed with a pivot table, grouping by depth, instance, and evaluation method.
4. The CDF of the advantage is plotted, per graph type and evaluation method.

In [16]:
instances_per_depth: dict[int, dict[str, set[str]]] = defaultdict(dict)
# We only go from depth 2 QAOA as this is what is plotted in the best methods figure.
for depth in range(2, 11):
    for evaluation in ["SV", "PP", "MPS (Aer)", "MPS (Quimb)"]:
        instances_per_depth[depth][evaluation] = set(
            df_best[
                df_best["method_label"].isin(["Linear Ramp*", "Interp.*"])
                & (df_best["depth"] == depth)
                & (df_best["evaluation_label"] == evaluation)
            ]["instance"].unique()
        )

In [17]:
df_all = db.only_best_parameters("config").to_dataframe()
df_comparison: pd.DataFrame | None = None
# We only go from depth 2 QAOA as this is what is plotted in the best methods figure.
for depth in range(2, 11):
    for evaluation in ["SV", "PP", "MPS (Aer)", "MPS (Quimb)"]:
        sub_df = df_all[
            (df_all["depth"] == depth)
            & (df_all["evaluation_label"] == evaluation)
            & (df_all["method_label"].isin(["Linear Ramp*", "Interp.*"]))
            & (df_all["instance"].isin(instances_per_depth[depth][evaluation]))
        ]
        if df_comparison is None:
            df_comparison = sub_df
        else:
            df_comparison = pd.concat([df_comparison, sub_df])
df_comparison

,instance,num_nodes,graph_type,trainer_config,method,depth,energy,trainer,evaluation,evaluation_label,...,metadata,result_index,run_datetime,result_key_index,approximation_ratio,mps_bond_dimension,mps_threshold,pp_max_weight,pp_min_abs_coeff,fa_degree
51,001_20nodes_random3regular.json,20,random_regular,I_SV_opt.json,I_opt.json,2,6.135491,ScipyTrainer,SV,SV,...,"{'iteration': '2', 'version': 28, 'evaluator':...",0,2026-01-29 15:44:26,2.2,NaN,NaN,NaN,NaN,NaN,NaN
80,001_20nodes_random3regular.json,20,random_regular,LR_SV_opt.json,LR_opt.json,2,7.128881,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 33, 'evaluator':...",0,2026-04-28 09:21:59,1.0,NaN,NaN,NaN,NaN,NaN,NaN
312,001_20nodes_random6regular.json,20,random_regular,I_SV_opt.json,I_opt.json,2,9.272494,ScipyTrainer,SV,SV,...,"{'iteration': '2', 'version': 19, 'evaluator':...",0,2025-12-01 09:01:12,2.2,NaN,NaN,NaN,NaN,NaN,NaN
361,001_20nodes_random6regular.json,20,random_regular,LR_SV_opt.json,LR_opt.json,2,9.339721,ScipyTrainer,SV,SV,...,"{'iteration': '1', 'version': 33, 'evaluator':...",0,2026-04-28 09:23:04,1.0,NaN,NaN,NaN,NaN,NaN,NaN
502,001_20nodes_random4regular.json,20,random_regular,I_SV_opt.json,I_opt.json,2,7.351810,ScipyTrainer,SV,SV,...,"{'iteration': '2', 'version': 19, 'evaluator':...",0,2025-12-01 06:31:21,2.2,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12233,007_50nodes_erdosrenyi20percent.json,50,erdos_renyi,LR_MPS_opt.json,LR_opt.json,10,51.137339,ScipyTrainer,MPS,MPS (Quimb),...,"{'iteration': '1', 'version': 33, 'evaluator':...",0,2026-04-18 04:26:35,1.0,NaN,16.0,1.000000e-10,NaN,NaN,NaN
12323,000_50nodes_erdosrenyi20percent.json,50,erdos_renyi,I_MPS_opt.json,I_opt.json,10,51.417922,RecursionTrainer,MPS,MPS (Quimb),...,"{'iteration': '2', 'version': 27, 'evaluator':...",0,2026-01-21 17:20:38,2.0,NaN,16.0,1.000000e-10,NaN,NaN,NaN
12521,000_50nodes_erdosrenyi20percent.json,50,erdos_renyi,LR_MPS_opt.json,LR_opt.json,10,50.497546,ScipyTrainer,MPS,MPS (Quimb),...,"{'iteration': '1', 'version': 33, 'evaluator':...",0,2026-04-18 04:26:49,1.0,NaN,16.0,1.000000e-10,NaN,NaN,NaN
12611,003_50nodes_erdosrenyi20percent.json,50,erdos_renyi,I_MPS_opt.json,I_opt.json,10,46.290480,RecursionTrainer,MPS,MPS (Quimb),...,"{'iteration': '2', 'version': 27, 'evaluator':...",0,2026-01-22 14:20:34,2.0,NaN,16.0,1.000000e-10,NaN,NaN,NaN


In [18]:
pivot_comparison = df_comparison.pivot_table(
    index=["instance", "depth", "evaluation_label", "graph_type"],
    columns="method_label",
    values="energy",
    aggfunc="mean",
)
pivot_comparison = pivot_comparison.reset_index()
pivot_comparison["Relative Advantage"] = (
    pivot_comparison["Linear Ramp*"] - pivot_comparison["Interp.*"]
) / pivot_comparison["Linear Ramp*"]
pivot_comparison["Graph Type"] = pivot_comparison["graph_type"].map(
    {
        "erdos_renyi": "ER",
        "heavy_hex": "HH",
        "random_regular": "RR",
        "line_to_full": "LB",
    }
)
pivot_comparison


method_label,instance,depth,evaluation_label,graph_type,Interp.*,Linear Ramp*,Relative Advantage,Graph Type
0,000_1_2_heavyhex_21nodes_weighted.json,2,SV,heavy_hex,5.068651,5.447847,0.069605,HH
1,000_1_2_heavyhex_21nodes_weighted.json,3,SV,heavy_hex,5.816423,5.836778,0.003487,HH
2,000_1_2_heavyhex_21nodes_weighted.json,4,SV,heavy_hex,6.207640,6.203019,-0.000745,HH
3,000_1_2_heavyhex_21nodes_weighted.json,5,SV,heavy_hex,6.393101,6.385080,-0.001256,HH
4,000_1_2_heavyhex_21nodes_weighted.json,6,SV,heavy_hex,6.536676,6.506042,-0.004709,HH
...,...,...,...,...,...,...,...,...
1020,009_7_3_heavyhex_144nodes_weighted.json,8,PP,heavy_hex,57.331558,56.221330,-0.019747,HH
1021,009_7_3_heavyhex_144nodes_weighted.json,9,MPS (Quimb),heavy_hex,33.085526,27.829724,-0.188856,HH
1022,009_7_3_heavyhex_144nodes_weighted.json,9,PP,heavy_hex,58.069562,56.786717,-0.022591,HH
1023,009_7_3_heavyhex_144nodes_weighted.json,10,MPS (Quimb),heavy_hex,33.728281,24.885210,-0.355354,HH


In [19]:
from matplotlib.lines import Line2D
import matplotlib.ticker as ticker
from scipy.optimize import minimize

In [20]:
fig, axes = plt.subplots(
    2,
    2,
    sharex=False,
    sharey=True,
    figsize=figsize(1.0),
    gridspec_kw=dict(hspace=0.4, wspace=0.1),
)
# Use different colors so we don't confuse them with methods. Use colours and
# ordering that matches arXiv paper.
palette = [plt.colormaps["Set1"](i) for i in [1, 0, 2, 3]]
graph_types = ["HH", "ER", "LB", "RR"]
for ax_label, ax, evaluation in zip(
    "abcdefghi", axes.flatten(), ["SV", "PP", "MPS (Aer)", "MPS (Quimb)"]
):
    print(f"Subplot ({ax_label}) {evaluation}")
    ax.set_title(f"({ax_label})", loc="left")
    sns.ecdfplot(
        pivot_comparison[pivot_comparison["evaluation_label"] == evaluation],
        x="Relative Advantage",
        hue="Graph Type",
        hue_order=graph_types,
        ax=ax,
        palette=palette,
        legend=False,
    )
    ax.set_xlabel("")
    ax.set_ylabel("")
    if evaluation == "SV":
        ax.xaxis.set_major_locator(ticker.MultipleLocator(0.2))
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.05))
    elif evaluation == "PP":
        _xlim = ax.get_xlim()
        ax.set_xlim(_xlim[0], _xlim[1] + 0.05)
        ax.xaxis.set_major_locator(ticker.MultipleLocator(0.2))
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.05))
    else:
        ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.25))

    # Add crossover values
    crossovers: dict[str, float] = {}
    for graph_type in graph_types:
        _sub_df = pivot_comparison[
            (pivot_comparison["Graph Type"] == graph_type)
            & (pivot_comparison["evaluation_label"] == evaluation)
        ]["Relative Advantage"]
        crossovers[graph_type] = len(_sub_df[_sub_df <= 0]) / len(_sub_df)
    ax.axvline(0, color="k", zorder=1)

    # Optimize text positions
    min_separation = 0.1
    upper_margin_separation = 0.07
    lower_margin_separation = 0.05
    ylim = ax.get_ylim()
    crossover_values = list(crossovers.values())

    def objective(positions):
        return np.sum(np.abs(positions - crossover_values))

    def constraint_separation(positions):
        """Ensure minimum separation between consecutive text positions"""
        sorted_positions = np.sort(positions)
        return sorted_positions[1:] - sorted_positions[:-1] - min_separation

    def constraint_lower_margin(positions):
        """Ensure minimum separation from lower y-axis limit"""
        return np.min(positions) - ylim[0] - lower_margin_separation

    def constraint_upper_margin(positions):
        """Ensure minimum separation from upper y-axis limit"""
        return ylim[1] - np.max(positions) - upper_margin_separation

    constraints = [
        {"type": "ineq", "fun": constraint_separation},
        {"type": "ineq", "fun": constraint_lower_margin},
        {"type": "ineq", "fun": constraint_upper_margin},
    ]

    result = minimize(
        objective,
        x0=crossover_values,
        method="SLSQP",
        constraints=constraints,
    )
    text_positions = result.x

    for i_graph_type, (graph_type, crossover) in enumerate(crossovers.items()):
        zero_xpos = ax.transAxes.inverted().transform(ax.transData.transform((0, 0)))[0]
        ax.axhline(
            crossover,
            color=palette[graph_types.index(graph_type)],
            linestyle=(i_graph_type, (1, 1)),
            xmin=zero_xpos if evaluation in ["SV", "PP"] else 0.25,
            xmax=zero_xpos if evaluation in ["MPS (Aer)", "MPS (Quimb)"] else 0.75,
            linewidth=1,
        )
        x_pos = 0.95 if evaluation in ["SV", "PP"] else 0.05
        x_axes = ax.transData.inverted().transform(ax.transAxes.transform((x_pos, 0)))[
            0
        ]
        ax.text(
            x_axes,
            text_positions[i_graph_type],
            f"${crossover * 100:.0f}\\%$",
            color=palette[graph_types.index(graph_type)],
            ha="right" if evaluation in ["SV", "PP"] else "left",
            va="center",
            fontsize="small",
        )
        ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
legend_entries = [
    (Line2D([], [], color=color), graph_type)
    for color, graph_type in zip(palette, graph_types)
]
fig.supxlabel("Relative Linear Ramp$^\\star$ Advantage")
fig.supylabel("Proportion of Graph-Depth Combinations", x=0)
fig.legend(
    *zip(*legend_entries),
    ncol=4,
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.0),
)
# plt.tight_layout()
fig.savefig("figures/figure19.pdf", dpi=300, bbox_inches="tight")

Subplot (a) SV
Subplot (b) PP
Subplot (c) MPS (Aer)
Subplot (d) MPS (Quimb)
